# CNN — Classification de panneaux de signalisation (GTSRB)
**Dataset** : German Traffic Sign Recognition Benchmark (43 classes, images 32×32)  
**Données utilisées** : `data3.pickle` — RGB, shuffle + normalisation /255 + mean + std

## 1. Imports

In [ ]:
import pickle
import csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tqdm import tqdm

# Reproductibilité
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilisé : {DEVICE}')

## 2. Chargement des données

In [ ]:
# Chargement de data3.pickle : meilleur préprocessing RGB
# (shuffle + /255 + soustraction moyenne + division std)
with open('data3.pickle', 'rb') as f:
    data = pickle.load(f, encoding='latin1')

x_train = data['x_train'].astype(np.float32)        # (N, 3, 32, 32)
y_train = data['y_train'].astype(np.int64)           # (N,)
x_val   = data['x_validation'].astype(np.float32)   # (M, 3, 32, 32)
y_val   = data['y_validation'].astype(np.int64)      # (M,)
x_test  = data['x_test'].astype(np.float32)          # (K, 3, 32, 32)
y_test  = data['y_test'].astype(np.int64)             # (K,)

print(f'Train      : {x_train.shape}  labels: {y_train.shape}')
print(f'Validation : {x_val.shape}  labels: {y_val.shape}')
print(f'Test       : {x_test.shape}  labels: {y_test.shape}')

In [ ]:
# Chargement des noms de classes
def load_label_names(file):
    names = []
    with open(file, 'r') as f:
        reader = csv.reader(f)
        for row in reader:
            names.append(row[1])
    return names[1:]  # supprimer l'en-tête

label_names = load_label_names('label_names.csv')
NUM_CLASSES = len(label_names)
print(f'Nombre de classes : {NUM_CLASSES}')
print('Exemples :', label_names[:5])

## 3. Exploration des données

In [ ]:
# Distribution des classes dans le jeu d'entraînement
counts = np.bincount(y_train)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(NUM_CLASSES), counts, color='steelblue', edgecolor='white')
ax.set_xlabel('Classe')
ax.set_ylabel('Nombre d\'exemples')
ax.set_title('Distribution des classes (train)')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(range(NUM_CLASSES), fontsize=7)
plt.tight_layout()
plt.show()

print(f'Min exemples par classe : {counts.min()} (classe {counts.argmin()})')
print(f'Max exemples par classe : {counts.max()} (classe {counts.argmax()})')

In [ ]:
# Affichage d'un exemple par classe
# Reconstruction approximative pour visualisation (dénormalisation)
with open('mean_image_rgb.pickle', 'rb') as f:
    mean_rgb = pickle.load(f, encoding='latin1')['mean_image_rgb']  # (3, 32, 32)
with open('std_rgb.pickle', 'rb') as f:
    std_rgb = pickle.load(f, encoding='latin1')['std_rgb']           # (3, 32, 32)

def denormalize(img):
    """img : (3, 32, 32) normalisé → uint8 (H, W, 3)"""
    img = img * std_rgb + mean_rgb
    img = img * 255.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img.transpose(1, 2, 0)  # (H, W, 3)

fig, axes = plt.subplots(5, 9, figsize=(16, 10))
axes = axes.ravel()

for cls in range(NUM_CLASSES):
    idx = np.where(y_train == cls)[0][0]
    img = denormalize(x_train[idx])
    axes[cls].imshow(img)
    axes[cls].set_title(f'{cls}', fontsize=7)
    axes[cls].axis('off')

# masquer les cases vides
for i in range(NUM_CLASSES, len(axes)):
    axes[i].axis('off')

fig.suptitle('Un exemple par classe', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Préparation des DataLoaders PyTorch

In [ ]:
BATCH_SIZE = 128

# Conversion en tenseurs
train_dataset = TensorDataset(
    torch.from_numpy(x_train),
    torch.from_numpy(y_train)
)
val_dataset = TensorDataset(
    torch.from_numpy(x_val),
    torch.from_numpy(y_val)
)
test_dataset = TensorDataset(
    torch.from_numpy(x_test),
    torch.from_numpy(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Batches train : {len(train_loader)}')
print(f'Batches val   : {len(val_loader)}')
print(f'Batches test  : {len(test_loader)}')

## 5. Architecture CNN

Architecture VGG-like légère adaptée aux images 32×32 :
- **Bloc 1** : Conv 3×3 (32) → BN → ReLU → Conv 3×3 (32) → BN → ReLU → MaxPool → Dropout
- **Bloc 2** : Conv 3×3 (64) → BN → ReLU → Conv 3×3 (64) → BN → ReLU → MaxPool → Dropout  
- **Bloc 3** : Conv 3×3 (128) → BN → ReLU → MaxPool → Dropout
- **Classifieur** : FC(2048→512) → ReLU → Dropout → FC(512→43)

In [ ]:
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()

        self.features = nn.Sequential(
            # Bloc 1 — sortie : (N, 32, 16, 16)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            # Bloc 2 — sortie : (N, 64, 8, 8)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            # Bloc 3 — sortie : (N, 128, 4, 4)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


model = TrafficSignCNN(num_classes=NUM_CLASSES).to(DEVICE)

# Résumé du modèle
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nParamètres entraînables : {total_params:,}')

## 6. Entraînement

In [ ]:
NUM_EPOCHS    = 30
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
# Réduit le LR si la val_loss ne s'améliore plus pendant 5 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)


def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.no_grad() if not train else torch.enable_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            if train:
                optimizer.zero_grad()

            logits = model(xb)
            loss   = criterion(logits, yb)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == yb).sum().item()
            total      += xb.size(0)

    return total_loss / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)

    scheduler.step(va_loss)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), 'best_model.pth')

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} '
          f'| train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} '
          f'| val_loss={va_loss:.4f}  val_acc={va_acc:.4f}')

print(f'\nMeilleure val_acc : {best_val_acc:.4f}')

## 7. Courbes d'apprentissage

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, history['train_acc'], label='Train')
ax2.plot(epochs, history['val_acc'],   label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Évaluation sur le jeu de test

In [ ]:
# Chargement du meilleur modèle
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        preds = model(xb).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = accuracy_score(all_labels, all_preds)
print(f'Accuracy sur le test : {test_acc:.4f} ({test_acc*100:.2f}%)')

In [ ]:
# Rapport de classification détaillé
print(classification_report(
    all_labels, all_preds,
    target_names=[f'{i}: {n[:20]}' for i, n in enumerate(label_names)],
    digits=3
))

## 9. Matrice de confusion

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
# Normalisation par ligne (recall par classe)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm_norm,
    annot=False,
    fmt='.2f',
    cmap='Blues',
    xticklabels=range(NUM_CLASSES),
    yticklabels=range(NUM_CLASSES),
    ax=ax,
    vmin=0, vmax=1
)
ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Vérité terrain', fontsize=12)
ax.set_title('Matrice de confusion normalisée (recall)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Visualisation des prédictions

In [ ]:
# 20 images aléatoires du test avec prédiction et vérité
rng   = np.random.default_rng(0)
idxs  = rng.choice(len(x_test), size=20, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
axes = axes.ravel()

for i, idx in enumerate(idxs):
    img   = denormalize(x_test[idx])
    truth = y_test[idx]
    pred  = all_preds[idx]

    axes[i].imshow(img)
    color = 'green' if truth == pred else 'red'
    axes[i].set_title(
        f'Vrai : {truth}\nPréd : {pred}',
        color=color, fontsize=8
    )
    axes[i].axis('off')

fig.suptitle('Prédictions sur le jeu de test (vert=correct, rouge=erreur)', fontsize=12)
plt.tight_layout()
plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Top-5 classes les plus difficiles

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)
worst_5 = np.argsort(per_class_acc)[:5]

print('Classes avec le plus faible recall :')
for cls in worst_5:
    print(f'  Classe {cls:2d} — {label_names[cls]:<40s} recall={per_class_acc[cls]:.3f}')

best_5 = np.argsort(per_class_acc)[-5:][::-1]
print('\nClasses avec le meilleur recall :')
for cls in best_5:
    print(f'  Classe {cls:2d} — {label_names[cls]:<40s} recall={per_class_acc[cls]:.3f}')

## Résumé

| Métrique | Valeur |
|---|---|
| Architecture | CNN VGG-like (3 blocs conv + BN + Dropout) |
| Paramètres | ~2.5M |
| Optimiseur | Adam + ReduceLROnPlateau |
| Données | data3.pickle (RGB, /255 + mean + std) |
| Classes | 43 |

Les fichiers générés :
- `best_model.pth` — poids du meilleur modèle
- `training_curves.png` — courbes loss / accuracy
- `confusion_matrix.png` — matrice de confusion
- `predictions.png` — exemples de prédictions